In [17]:
import json
import torch
from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
import warnings
warnings.filterwarnings("ignore")

In [18]:
dataset_json = [] 
import json

try:
    with open('data.json', 'r') as file:
        dataset_json = json.load(file)
    print("JSON data from file:")
except FileNotFoundError:
    print("Error: The file 'data.json' was not found.")
except json.JSONDecodeError:
    print("Error: Failed to decode JSON from the file (malformed JSON).")


JSON data from file:


In [19]:
from datasets import Dataset
import pandas as pd
from sklearn.preprocessing import LabelEncoder

# Load CSV
df = pd.read_csv("data_01_labeled.csv")

texts = []
labels = []

def is_valid_text(text):
    if pd.isna(text):
        return False
    if str(text).strip() == "":
        return False
    return True

# Extract dataset
for _, row in df.iterrows():

    if is_valid_text(row["Experience"]):
        texts.append(str(row["Experience"]).strip())
        labels.append("Experience")

    if is_valid_text(row["Skill"]):
        texts.append(str(row["Skill"]).strip())
        labels.append("Skill")

    if is_valid_text(row["Qualification"]):
        texts.append(str(row["Qualification"]).strip())
        labels.append("Qualification")

# Label Encoding
le = LabelEncoder()
labels_encoded = le.fit_transform(labels)

print("Label Mapping:")
for label, index in zip(le.classes_, range(len(le.classes_))):
    print(f"{label} → {index}")

# Create HuggingFace Dataset
dataset = Dataset.from_dict({
    "text": texts,
    "label": labels_encoded
})

Label Mapping:
Experience → 0
Qualification → 1
Skill → 2


In [20]:
import joblib

# Save label encoder
joblib.dump(le, "label_encoder.pkl")

['label_encoder.pkl']

In [21]:
# Convert to Dataset
# texts = [item["text"] for item in dataset_json]
# labels = [item["label"] for item in dataset_json]

# Encode labels to integers
# le = LabelEncoder()
# labels_encoded = le.fit_transform(labels)

In [22]:
# Train/test split

from sklearn.model_selection import train_test_split

# Step 1: 70% train, 30% temp
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    texts,
    labels_encoded,
    test_size=0.30,
    stratify=labels_encoded,
    random_state=42
)

# Step 2: Split temp into 15% val and 15% test
val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts,
    temp_labels,
    test_size=0.50,
    stratify=temp_labels,
    random_state=42
)

print("Train size:", len(train_texts))
print("Validation size:", len(val_texts))
print("Test size:", len(test_texts))


train_dataset = Dataset.from_dict({"text": train_texts, "label": train_labels})
val_dataset = Dataset.from_dict({"text": val_texts, "label": val_labels})

Train size: 16326
Validation size: 3499
Test size: 3499


In [23]:
# ========================
# Step 2: Tokenization
# ========================
model_name = "prajjwal1/bert-tiny"  # lightweight and fast
tokenizer = AutoTokenizer.from_pretrained(model_name)

def tokenize(batch):
    return tokenizer(batch["text"], padding='max_length', truncation=True, max_length=128)

train_dataset = train_dataset.map(tokenize, batched=True)
val_dataset = val_dataset.map(tokenize, batched=True)

# Set format for PyTorch
train_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])
val_dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "label"])

Map: 100%|████████████████████████████████████████████████████████████████| 3499/3499 [00:01<00:00, 1895.29 examples/s]


In [24]:


from transformers import AutoModelForSequenceClassification

num_labels = len(le.classes_)

# This will print progress while downloading/loading
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=num_labels,
    force_download=False,  # Set True if you want to re-download
    local_files_only=False  # Allow download if not cached
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at prajjwal1/bert-tiny and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [15]:
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(pred):
    logits, labels = pred
    predictions = logits.argmax(axis=-1)

    precision, recall, f1_weighted, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted"
    )

    _, _, f1_macro, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro"
    )

    acc = accuracy_score(labels, predictions)

    return {
        "accuracy": acc,
        "f1": f1_weighted,
        "f1_macro": f1_macro,
        "precision": precision,
        "recall": recall
    }

In [20]:
training_args = TrainingArguments(
    output_dir="./cv_classifier_model",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    logging_dir="./logs",
    learning_rate=2e-5,
    eval_strategy='epoch', 
    save_strategy='epoch',
    metric_for_best_model="eval_f1", 
    greater_is_better=True, 
    weight_decay=0.01, 
    logging_steps=50,
    save_safetensors=False,
    load_best_model_at_end=True  # no need for best model
)


In [21]:
# ========================
# Step 6: Trainer
# ========================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
)

In [22]:
# ========================
# Step 7: Train
# ========================
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy,F1,F1 Macro,Precision,Recall
1,0.029100,0.015052,0.999714,0.999714,0.999717,0.999714,0.999714
2,0.004200,0.004956,0.999714,0.999714,0.999717,0.999714,0.999714
3,0.001700,0.001751,1.000000,1.000000,1.000000,1.000000,1.000000
4,0.000900,0.000833,1.000000,1.000000,1.000000,1.000000,1.000000
5,0.000700,0.000632,1.000000,1.000000,1.000000,1.000000,1.000000


TrainOutput(global_step=10205, training_loss=0.04550221149692388, metrics={'train_runtime': 1604.5108, 'train_samples_per_second': 50.875, 'train_steps_per_second': 6.36, 'total_flos': 25935551516160.0, 'train_loss': 0.04550221149692388, 'epoch': 5.0})

In [74]:
# ========================
# Step 9: Example Inference
# ========================
def predict(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True, max_length=128)
    outputs = model(**inputs)
    preds = torch.argmax(outputs.logits, dim=1).item()
    label = le.inverse_transform([preds])[0]
    return label

# Example prediction
example_text = "Bachelor's in Computer Engineering University of Pune "
print("Predicted Label:", predict(example_text))

Predicted Label: Qualification


In [7]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

model = AutoModelForSequenceClassification.from_pretrained("./cv_classifier_model")
tokenizer = AutoTokenizer.from_pretrained("./cv_classifier_model")


## Upload CV and Classify the data 

In [2]:

def classify_line(line):
    inputs = tokenizer(line, return_tensors="pt", truncation=True)
    outputs = model(**inputs)
    return outputs.logits.argmax(dim=1).item()


In [3]:
import pdfplumber

def extract_text_from_pdf(pdf_path):
    text = ""
    with pdfplumber.open(pdf_path) as pdf:
        for page in pdf.pages:
            text += page.extract_text()
    return text

cv_text = extract_text_from_pdf("resume_pdfs/resume_00004_Python_Developer.pdf")


In [4]:
import re
import string

def clean_text(text):
    text = text.lower()  # lowercase
    text = re.sub(r'\S+@\S+', '', text)  # remove email
    text = re.sub(r'\d+', '', text)  # remove numbers
    text = text.translate(str.maketrans('', '', string.punctuation))  # remove punctuation
    return text

cleaned_cv = clean_text(cv_text)


In [5]:
sentences = cleaned_cv.split("\n")

In [6]:
from collections import defaultdict
import torch

result = defaultdict(list)

for sentence in sentences:
    cleaned = clean_text(sentence)

    inputs = tokenizer(cleaned, return_tensors="pt", truncation=True, padding=True)
    
    with torch.no_grad():
        outputs = model(**inputs)

    prediction = outputs.logits.argmax(dim=1).item()
    
    label = le.inverse_transform([prediction])[0]   # get string label
    
    result[label].append(sentence)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


NameError: name 'le' is not defined

In [ ]:
result['Experience']